# 02. Segment Tree

A segment tree answers **range queries** (sum, min, max, gcd) and supports **point updates** and even **range updates** (lazy propagation), all in O(log n).

**Learning order:** REAL LIFE → STRUCTURE → ITERATIVE VERSION → RANGE MINIMUM → LAZY PROPAGATION → WHEN TO USE → INTERVIEW THINKING

## 1. Real life

A weather app: "lowest temperature between day 10 and day 200", while day 57's reading can change. A game: "total damage dealt by players 20 to 50". A segment tree splits the data into halves repeatedly, so any range is covered by at most `2 log n` nodes.

**Recognition clue:** "range min/max/sum with updates", "range update", "number of elements in range".

## 2. Structure

A complete binary tree where each node stores the **answer for a segment** of the array. The root covers `[0, n-1]`, its children cover the two halves, leaves cover one element.

- **Build:** O(n)
- **Point update:** change the leaf, recompute the parents on the path: O(log n)
- **Range query:** combine only the nodes fully inside the range: O(log n)
- **Space:** O(n) (about 2n with the bottom-up layout, 4n for the recursive layout)

In [1]:
class SegmentTree:
    """Iterative (bottom-up) segment tree. `op` must be associative (sum, min, max, gcd, ...)."""
    def __init__(self, data, op, identity):
        self.n, self.op, self.identity = len(data), op, identity
        self.tree = [identity] * self.n + list(data)          # leaves at n..2n-1
        for i in range(self.n - 1, 0, -1):
            self.tree[i] = op(self.tree[2 * i], self.tree[2 * i + 1])

    def update(self, i, value):                               # set data[i] = value
        i += self.n
        self.tree[i] = value
        while i > 1:
            i //= 2
            self.tree[i] = self.op(self.tree[2 * i], self.tree[2 * i + 1])

    def query(self, l, r):                                    # inclusive range [l, r]
        res_left = res_right = self.identity
        l += self.n
        r += self.n + 1
        while l < r:
            if l & 1:
                res_left = self.op(res_left, self.tree[l]); l += 1
            if r & 1:
                r -= 1; res_right = self.op(self.tree[r], res_right)
            l //= 2
            r //= 2
        return self.op(res_left, res_right)

st = SegmentTree([2, 5, 1, 4, 9, 3], min, float("inf"))
print(st.query(1, 4))   # 1
st.update(2, 7)
print(st.query(1, 4))   # 4

sm = SegmentTree([2, 5, 1, 4, 9, 3], lambda a, b: a + b, 0)
print(sm.query(0, 5), sm.query(2, 3))   # 24 5

1
4
24 5


## 3. Why queries are O(log n)

At each level, at most **two** nodes are taken. The range is split into `O(log n)` canonical segments. That is why sum, min and max all work the same way: only the `op` changes.

## 4. Lazy propagation: range **updates**

"Add `x` to every element in `[l, r]`" would cost O(n) if done element by element. With **lazy propagation** we store a pending update (`lazy`) on a node and push it down to the children **only when we need to go deeper**. Range update and range query are both O(log n).

Below: **range add + range sum**. A node covering `k` elements, when added `x`, increases its sum by `x * k`.

In [2]:
class LazySegmentTree:
    def __init__(self, data):
        self.n = len(data)
        self.sum = [0] * (4 * self.n)
        self.lazy = [0] * (4 * self.n)
        self._build(1, 0, self.n - 1, data)

    def _build(self, node, lo, hi, data):
        if lo == hi:
            self.sum[node] = data[lo]
            return
        mid = (lo + hi) // 2
        self._build(2 * node, lo, mid, data)
        self._build(2 * node + 1, mid + 1, hi, data)
        self.sum[node] = self.sum[2 * node] + self.sum[2 * node + 1]

    def _push(self, node, lo, hi):                 # send the pending add to the children
        if self.lazy[node]:
            mid = (lo + hi) // 2
            for child, clo, chi in ((2 * node, lo, mid), (2 * node + 1, mid + 1, hi)):
                self.sum[child] += self.lazy[node] * (chi - clo + 1)
                self.lazy[child] += self.lazy[node]
            self.lazy[node] = 0

    def add(self, l, r, x, node=1, lo=0, hi=None):
        if hi is None:
            hi = self.n - 1
        if r < lo or hi < l:
            return
        if l <= lo and hi <= r:                    # fully covered: stop here, remember lazily
            self.sum[node] += x * (hi - lo + 1)
            self.lazy[node] += x
            return
        self._push(node, lo, hi)
        mid = (lo + hi) // 2
        self.add(l, r, x, 2 * node, lo, mid)
        self.add(l, r, x, 2 * node + 1, mid + 1, hi)
        self.sum[node] = self.sum[2 * node] + self.sum[2 * node + 1]

    def query(self, l, r, node=1, lo=0, hi=None):
        if hi is None:
            hi = self.n - 1
        if r < lo or hi < l:
            return 0
        if l <= lo and hi <= r:
            return self.sum[node]
        self._push(node, lo, hi)
        mid = (lo + hi) // 2
        return self.query(l, r, 2 * node, lo, mid) + self.query(l, r, 2 * node + 1, mid + 1, hi)

lz = LazySegmentTree([1, 2, 3, 4, 5])
lz.add(1, 3, 2)                       # [1, 4, 5, 6, 5]
print(lz.query(0, 4), lz.query(1, 2))   # 21 9

21 9


## 5. When to use which structure

| Need | Choose |
|---|---|
| range sum, point updates | Fenwick (shorter code) or segment tree |
| range **min/max** with updates | segment tree |
| range **updates** + range query | lazy segment tree (or Fenwick with 2 trees for add/sum) |
| static data, min/max queries | **sparse table** (next notebook) |
| offline queries | Mo's algorithm / sorting queries |

## 6. Interview thinking

**Say:** "A naive solution is O(n) per query/update. A segment tree splits the array so each operation touches O(log n) nodes. Build O(n), query O(log n), update O(log n)."

**Common mistakes:** array size too small for the recursive layout (`4n`), forgetting to push lazy values before recursing, off-by-one in inclusive/exclusive ranges, using a non-associative `op`, wrong identity (`inf` for min, `0` for sum).

## Quick Revision

- Segment tree: node = answer for a segment, build O(n), update/query O(log n).
- Works for any associative operation. Identity: `0` for sum, `inf` for min.
- Lazy propagation: store pending range updates, push down when needed.
- Iterative version: leaves at `n..2n-1`, parent `i // 2`.